### 📦 Part A–B: Dataset Construction & Splitting

Loads the patent claims dataset `AI-Growth-Lab/patents_claims_1.5m_traim_test` and extracts the relevant columns (`id`, `date`, `text`, and all `Y02*` CPC indicators).

Creates a **silver green label (`is_green_silver`)** using a simple heuristic: a claim is considered green if **any Y02 classification flag is present**.

Builds a **balanced 50k dataset** by sampling:
- 25k green claims  
- 25k non-green claims  

The dataset is then **stratified into three splits**:

- `train_silver` (70%) — training data for model adaptation  
- `pool_unlabeled` (20%) — pool used for uncertainty sampling  
- `eval_silver` (10%) — evaluation set  

Finally, the dataset is cleaned (`id → doc_id`), reduced to the required columns, and saved as:

`patents_50k_green.parquet`

This file is used as the **base dataset for active learning, agent labeling, and final model training**.

In [ ]:
# Create patents_50k_green.parquet from AI-Growth-Lab/patents_claims_1.5m_traim_test
# Balanced 25k green (Y02*) + 25k not green (no Y02*)
# Splits: train_silver / pool_unlabeled / eval_silver

# --- Imports: numeric ops + dataframe ops + HF datasets + stratified splitting ---
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# --- Dataset + output settings ---
DS_NAME = "AI-Growth-Lab/patents_claims_1.5m_traim_test"
OUT_PATH = "patents_50k_green.parquet"

# --- Reproducibility + sampling size per class ---
SEED = 42
N_PER_CLASS = 25_000

# 1) Load dataset (train split) and only needed columns
# --- Load HF dataset split (contains claim-level rows) ---
ds = load_dataset(DS_NAME, split="train")

# Identify Y02* columns from feature names
# --- Auto-detect all CPC Y02* indicator columns (green tech taxonomy) ---
cols = ds.column_names
y02_cols = [c for c in cols if c.startswith("Y02")]
print("Found Y02 columns:", y02_cols)

# keep minimal columns: id, date, text, and all Y02*
# --- Reduce memory: keep only id/date/text + Y02 indicator columns ---
keep_cols = ["id", "date", "text"] + y02_cols
ds_small = ds.select_columns(keep_cols)

# 2) Convert to pandas
# --- Convert to pandas for easier filtering/sampling/splitting ---
df = ds_small.to_pandas()

# 3) Create silver label:
# green if ANY Y02* column == 1
# (works whether columns are int/bool)
# --- Silver label heuristic: any Y02* flag => green (binary) ---
df["is_green_silver"] = (df[y02_cols].sum(axis=1) > 0).astype(int)

# --- Sanity check: class balance before sampling ---
print(df["is_green_silver"].value_counts())

# 4) Sample balanced 50k
# --- Balanced sampling: 25k positives + 25k negatives for assignment dataset ---
df_green = df[df["is_green_silver"] == 1].sample(N_PER_CLASS, random_state=SEED)
df_not = df[df["is_green_silver"] == 0].sample(N_PER_CLASS, random_state=SEED)

# --- Combine and shuffle so ordering is random ---
df_50k = pd.concat([df_green, df_not], ignore_index=True).sample(frac=1, random_state=SEED)

# 5) Create splits (stratified)
# --- Create three splits with stratification on silver label (keeps same class ratio per split) ---
train_df, temp_df = train_test_split(
    df_50k,
    test_size=0.30,
    stratify=df_50k["is_green_silver"],
    random_state=SEED
)
pool_df, eval_df = train_test_split(
    temp_df,
    test_size=1/3,  # 10% of total (because temp is 30%)
    stratify=temp_df["is_green_silver"],
    random_state=SEED
)

# --- Copy to avoid chained assignment pitfalls later ---
train_df = train_df.copy()
pool_df = pool_df.copy()
eval_df = eval_df.copy()

# --- Assign split labels used downstream in training/AL pipeline ---
train_df["split"] = "train_silver"
pool_df["split"]  = "pool_unlabeled"
eval_df["split"]  = "eval_silver"

# --- Merge splits back into one parquet for convenient loading/filtering later ---
final_df = pd.concat([train_df, pool_df, eval_df], ignore_index=True)

# 6) Rename id -> doc_id to match assignment wording
# --- Align column naming with assignment spec (doc_id) ---
final_df = final_df.rename(columns={"id": "doc_id"})

# 7) Keep only columns needed downstream
# --- Keep only what later scripts need: id/date/text/label/split ---
final_df = final_df[["doc_id", "date", "text", "is_green_silver", "split"]]

# --- Sanity checks: split counts and label counts ---
print(final_df["split"].value_counts())
print(final_df["is_green_silver"].value_counts())
print(final_df.head())

# 8) Save parquet
# --- Persist as parquet (fast + compact) for later training and sampling steps ---
final_df.to_parquet(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

Found Y02 columns: ['Y02A', 'Y02B', 'Y02C', 'Y02D', 'Y02E', 'Y02P', 'Y02T', 'Y02W']
is_green_silver
0    1269218
1     103692
Name: count, dtype: int64
split
train_silver      35000
pool_unlabeled    10000
eval_silver        5000
Name: count, dtype: int64
is_green_silver
1    25000
0    25000
Name: count, dtype: int64
    doc_id        date                                               text  \
0  9057357  2015-06-16  1. A wind turbine comprising: a first ring com...   
1  8709670  2014-04-29  1. A vehicle comprising: a fuel cell system co...   
2  9698720  2017-07-04  1. An electrical drive system, comprising: an ...   
3  9828531  2017-11-28  1. An adhesive composition comprising a contin...   
4  9028842  2015-05-12  1. A composition suitable for injection into a...   

   is_green_silver         split  
0                1  train_silver  
1                1  train_silver  
2                1  train_silver  
3                0  train_silver  
4                1  train_silver  
Saved: 

### Baseline Model — Frozen PatentSBERTa

Loads the prepared dataset (`patents_50k_green.parquet`) and uses **PatentSBERTa as a frozen feature extractor** to generate semantic embeddings for each patent claim.

Each claim is tokenized and passed through the transformer encoder. Token embeddings are aggregated using **masked mean pooling** and **L2-normalized** to produce a fixed-length sentence embedding.

Embeddings are computed in batches and **cached to disk** to avoid recomputation during later experiments.

A **Logistic Regression classifier** is then trained on the `train_silver` split using these frozen embeddings as features.  
The model is evaluated on the `eval_silver` split using **precision, recall, and F1-score**.

This baseline represents the **“Frozen Embeddings (No Fine-tuning)”** model used later for comparison in the final evaluation table.

The trained classifier, metadata, and embedding cache are saved to disk for reproducibility.

In [ ]:
# Frozen PatentSBERTa embeddings + Logistic Regression

# --- Imports: IO + arrays/dataframes + HF encoder + sklearn classifier + metrics + serialization ---
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, classification_report
import joblib

# ---------------- Config ----------------
# --- Data + model identifiers ---
PARQUET_PATH = "patents_50k_green.parquet"
MODEL_NAME = "AI-Growth-Lab/PatentSBERTa"

# --- Column names (kept consistent across scripts) ---
TEXT_COL = "text"
LABEL_COL = "is_green_silver"
SPLIT_COL = "split"
ID_COL = "doc_id"

# --- Embedding settings (encoder is frozen; only affects runtime/memory) ---
MAX_LENGTH = 256
BATCH_SIZE = 64 
OUT_DIR = "models_baseline"
CACHE_DIR = os.path.join(OUT_DIR, "emb_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Device (MPS first) ----------------
# --- Device selection: prefer Apple MPS, else CUDA, else CPU ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)


def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    # --- Sentence embedding via masked mean pooling over token embeddings ---
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [B,T,1]
    summed = (last_hidden_state * mask).sum(dim=1)                  # [B,H]
    counts = mask.sum(dim=1).clamp(min=1e-9)                        # [B,1]
    return summed / counts


@torch.no_grad()
def embed_texts(texts, tokenizer, model, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, device=DEVICE):
    # --- Batch inference: encode texts -> transformer -> pooled sentence embeddings ---
    model.eval()
    all_embs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i + batch_size]

        # --- Tokenization with padding/truncation to fixed max_length ---
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        # move to device
        enc = {k: v.to(device) for k, v in enc.items()}

        # --- Forward pass through frozen encoder ---
        out = model(**enc)
        token_embs = out.last_hidden_state  # [B,T,H]
        sent_embs = mean_pooling(token_embs, enc["attention_mask"])  # [B,H]

        # L2 normalize helps linear classifier
        # --- Normalize embeddings to unit length (often improves linear separability) ---
        sent_embs = torch.nn.functional.normalize(sent_embs, p=2, dim=1)

        all_embs.append(sent_embs.detach().cpu().numpy())

    return np.vstack(all_embs)


def load_or_compute_embeddings(cache_prefix, texts, tokenizer, model):
    # --- Simple disk cache: avoid recomputing expensive embeddings between runs ---
    x_path = os.path.join(CACHE_DIR, f"{cache_prefix}_X.npy")
    if os.path.exists(x_path):
        print(f"Loading cached embeddings: {x_path}")
        return np.load(x_path)

    X = embed_texts(texts, tokenizer, model)
    np.save(x_path, X)
    print(f"Saved embeddings: {x_path}")
    return X


def prf1(y_true, y_pred):
    # --- Convenience wrapper: binary precision/recall/F1 ---
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return p, r, f1


def main():
    # 1) Load parquet
    # --- Load the prepared 50k parquet and validate expected columns ---
    df = pd.read_parquet(PARQUET_PATH)

    # sanity checks
    for col in [TEXT_COL, LABEL_COL, SPLIT_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}'. Found columns: {list(df.columns)}")

    # --- Split: train_silver used for fitting; eval_silver used for evaluation ---
    train_df = df[df[SPLIT_COL] == "train_silver"].dropna(subset=[TEXT_COL, LABEL_COL]).copy()
    eval_df  = df[df[SPLIT_COL] == "eval_silver"].dropna(subset=[TEXT_COL, LABEL_COL]).copy()

    # --- Extract raw texts + labels ---
    X_train_text = train_df[TEXT_COL].astype(str).tolist()
    y_train = train_df[LABEL_COL].astype(int).to_numpy()

    X_eval_text = eval_df[TEXT_COL].astype(str).tolist()
    y_eval = eval_df[LABEL_COL].astype(int).to_numpy()

    # --- Basic dataset sanity: sizes and class balance ---
    print(f"Train size: {len(train_df)} | Eval size: {len(eval_df)}")
    print("Train label balance:", train_df[LABEL_COL].value_counts(normalize=True).to_dict())
    print("Eval label balance: ", eval_df[LABEL_COL].value_counts(normalize=True).to_dict())

    # 2) Load transformer (frozen)
    # --- Load PatentSBERTa encoder; no fine-tuning here (feature extractor baseline) ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)

    # IMPORTANT for Apple MPS stability: keep float32
    # --- Move model to device and keep eval() (no grads) ---
    model = model.to(DEVICE)
    model.eval()

    # 3) Embeddings (cached)
    # --- Compute (or load) embeddings for train and eval splits ---
    X_train = load_or_compute_embeddings("train_silver", X_train_text, tokenizer, model)
    X_eval  = load_or_compute_embeddings("eval_silver", X_eval_text, tokenizer, model)

    print("Embeddings shapes:", X_train.shape, X_eval.shape)

    # 4) Train baseline classifier
    # --- Simple linear classifier on top of frozen embeddings ---
    clf = LogisticRegression(
        max_iter=3000,
        n_jobs=-1,
        solver="lbfgs",
    )
    clf.fit(X_train, y_train)

    # 5) Evaluate
    # --- Predict and compute PRF1 on eval_silver ---
    y_pred = clf.predict(X_eval)
    p, r, f1 = prf1(y_eval, y_pred)

    print("\n=== Part A: Baseline results on eval_silver ===")
    print(f"Precision: {p:.4f}")
    print(f"Recall:    {r:.4f}")
    print(f"F1:        {f1:.4f}")
    print("\nClassification report:")
    print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

    # 6) Save baseline model + meta
    # --- Persist sklearn classifier + metadata so results are reproducible ---
    joblib.dump(clf, os.path.join(OUT_DIR, "baseline_logreg.joblib"))
    meta = {
        "transformer_model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "pooling": "mean_pooling + l2_normalize",
        "batch_size": BATCH_SIZE,
        "device_used": DEVICE,
        "features_dim": int(X_train.shape[1]),
        "train_rows": int(len(train_df)),
        "eval_rows": int(len(eval_df)),
        "parquet_path": PARQUET_PATH,
    }
    joblib.dump(meta, os.path.join(OUT_DIR, "baseline_meta.joblib"))

    print(f"\nSaved baseline classifier to: {OUT_DIR}/baseline_logreg.joblib")
    print(f"Saved meta to:               {OUT_DIR}/baseline_meta.joblib")
    print(f"Embeddings cached in:        {CACHE_DIR}/ (X.npy files)")


# --- Script entry point ---
if __name__ == "__main__":
    main()

Using device: mps
Train size: 35000 | Eval size: 5000
Train label balance: {1: 0.5, 0: 0.5}
Eval label balance:  {1: 0.5, 0: 0.5}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: AI-Growth-Lab/PatentSBERTa
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding: 100%|██████████| 547/547 [18:30<00:00,  2.03s/it]


Saved embeddings: models_baseline/emb_cache/train_silver_X.npy


Embedding: 100%|██████████| 79/79 [03:01<00:00,  2.30s/it]

Saved embeddings: models_baseline/emb_cache/eval_silver_X.npy
Embeddings shapes: (35000, 768) (5000, 768)



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISMhuggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process


=== Part A: Baseline results on eval_silver ===
Precision: 0.7813
Recall:    0.7432
F1:        0.7618

Classification report:
              precision    recall  f1-score   support

           0     0.7551    0.7920    0.7731      2500
           1     0.7813    0.7432    0.7618      2500

    accuracy                         0.7676      5000
   macro avg     0.7682    0.7676    0.7675      5000
weighted avg     0.7682    0.7676    0.7675      5000


Saved baseline classifier to: models_baseline/baseline_logreg.joblib
Saved meta to:               models_baseline/baseline_meta.joblib
Embeddings cached in:        models_baseline/emb_cache/ (X.npy files)


/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### Uncertainty Sampling — Top-100 High-Risk Claims

Loads the `pool_unlabeled` split from `patents_50k_green.parquet` and uses the **baseline Logistic Regression model** (trained on frozen PatentSBERTa embeddings) to score each claim with `p_green = P(green=1)`.

Embeddings for the pool are computed using the same **frozen PatentSBERTa encoder + mean pooling + L2 normalization** as in the baseline step, and are **cached** (`pool_unlabeled_X.npy`) for faster re-runs.

Uncertainty is computed as:

`u = 1 - 2 * |p_green - 0.5|`

This prioritizes claims where the classifier is **closest to 0.5** (most unsure). The script then selects the **top 100 most uncertain claims**.

Exports `hitl_green_100_unlabeled.csv` containing:
- `doc_id`, `text`
- baseline scores: `p_green`, `u`
- empty placeholder columns for downstream **MAS labeling + targeted HITL** (suggestion, confidence, rationale, human label, notes)

This CSV is the **fixed high-risk subset** used in the next steps of the pipeline.

In [ ]:
# Uncertainty sampling on pool_unlabeled
# Uses baseline_logreg + frozen PatentSBERTa embeddings
# Exports hitl_green_100.csv with required columns

# --- Imports: IO + arrays/dataframes + HF encoder + joblib loading ---
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import joblib

# ---------------- Config ----------------
# --- Data/model paths ---
PARQUET_PATH = "patents_50k_green.parquet"
MODEL_NAME = "AI-Growth-Lab/PatentSBERTa"

# --- Baseline classifier artifacts (trained earlier on train_silver embeddings) ---
BASELINE_PATH = "models_baseline/baseline_logreg.joblib"
META_PATH = "models_baseline/baseline_meta.joblib"
CACHE_DIR = "models_baseline/emb_cache"

# --- Output: top-100 uncertain claims for downstream MAS/HITL ---
OUT_CSV = "hitl_green_100_unlabeled.csv" # give the file name a different ending than the ones being used later as to not overwrite it by acident

# --- Column names ---
TEXT_COL = "text"
SPLIT_COL = "split"
ID_COL = "doc_id"

# --- Embedding runtime params (same as baseline for consistency) ---
MAX_LENGTH = 256
BATCH_SIZE = 64

# ---------------- Device (MPS first) ----------------
# --- Device selection: prefer MPS, else CUDA, else CPU ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    # --- Masked mean pooling to get a single vector per text ---
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_texts(texts, tokenizer, model, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, device=DEVICE):
    # --- Batch embedding for pool texts (frozen encoder; no gradients) ---
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding pool_unlabeled"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        # --- Move tokenized inputs to accelerator ---
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        # --- Pool token embeddings -> sentence embeddings, then L2 normalize ---
        sent_embs = mean_pooling(out.last_hidden_state, enc["attention_mask"])
        sent_embs = torch.nn.functional.normalize(sent_embs, p=2, dim=1)
        all_embs.append(sent_embs.cpu().numpy())
    return np.vstack(all_embs)


def load_or_compute_pool_embeddings(pool_texts, tokenizer, model):
    # --- Cache embeddings for pool_unlabeled to avoid re-embedding across runs ---
    x_path = os.path.join(CACHE_DIR, "pool_unlabeled_X.npy")
    if os.path.exists(x_path):
        print(f"Loading cached pool embeddings: {x_path}")
        return np.load(x_path)
    X = embed_texts(pool_texts, tokenizer, model)
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(x_path, X)
    print(f"Saved pool embeddings: {x_path}")
    return X


def main():
    # 1) Load data
    # --- Load parquet and filter to unlabeled pool split (AL candidate set) ---
    df = pd.read_parquet(PARQUET_PATH)
    pool_df = df[df[SPLIT_COL] == "pool_unlabeled"].dropna(subset=[TEXT_COL]).copy()

    # --- Ensure IDs exist (needed to trace back to rows later) ---
    if ID_COL not in pool_df.columns:
        raise ValueError(f"Expected '{ID_COL}' in parquet. Found: {list(pool_df.columns)}")

    pool_texts = pool_df[TEXT_COL].astype(str).tolist()
    print("Pool size:", len(pool_df))

    # 2) Load baseline model
    # --- Load trained LogisticRegression baseline + meta info for sanity check ---
    clf = joblib.load(BASELINE_PATH)
    meta = joblib.load(META_PATH)
    print("Loaded baseline + meta:", meta.get("transformer_model", "n/a"))

    # 3) Load transformer for embeddings (frozen)
    # --- Load same encoder used in baseline feature extraction ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    # 4) Get embeddings for pool_unlabeled (cached)
    # --- Compute or reuse pool embeddings for consistent probability estimates ---
    X_pool = load_or_compute_pool_embeddings(pool_texts, tokenizer, model)

    # 5) Predict probabilities
    # --- Use baseline classifier to get p(green=1) for each pool item ---
    # predict_proba returns [p(class0), p(class1)] where class1 should be "green=1"
    proba = clf.predict_proba(X_pool)
    p_green = proba[:, 1]

    # 6) Uncertainty score u = 1 - 2*|p-0.5|
    # --- Uncertainty sampling: highest near 0.5, lowest near 0 or 1 ---
    u = 1.0 - 2.0 * np.abs(p_green - 0.5)

    # --- Attach model outputs to dataframe for sorting/debugging ---
    pool_df["p_green"] = p_green
    pool_df["u"] = u

    # 7) Select top 100 most uncertain
    # --- Take the 100 items the baseline is most unsure about (high-risk for labeling errors) ---
    top100 = pool_df.sort_values("u", ascending=False).head(100).copy()

    # 8) Prepare HITL CSV with required + empty labeling cols
    # --- Export with placeholders used in later MAS + human review workflow ---
    out = pd.DataFrame({
        "doc_id": top100[ID_COL].values,
        "text": top100[TEXT_COL].values,
        "p_green": top100["p_green"].values,
        "u": top100["u"].values,
        "llm_green_suggested": ["" for _ in range(len(top100))],
        "llm_confidence": ["" for _ in range(len(top100))],
        "llm_rationale": ["" for _ in range(len(top100))],
        "is_green_human": ["" for _ in range(len(top100))],
        "notes": ["" for _ in range(len(top100))],
    })

    out.to_csv(OUT_CSV, index=False)
    print(f"Saved HITL file: {OUT_CSV}")
    print(out.head(3))


# --- Script entry point ---
if __name__ == "__main__":
    main()

Using device: mps
Pool size: 10000
Loaded baseline + meta: AI-Growth-Lab/PatentSBERTa


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: AI-Growth-Lab/PatentSBERTa
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding pool_unlabeled: 100%|██████████| 157/157 [04:54<00:00,  1.88s/it]

Saved pool embeddings: models_baseline/emb_cache/pool_unlabeled_X.npy
Saved HITL file: hitl_green_100_unlabeled.csv
    doc_id                                               text   p_green  \
0  8593474  1. A method for reading data from a tiled orga...  0.500015   
1  9617453  1. A solvent-free aqueous polyurethane dispers...  0.499975   
2  8618928  1. A method for wireless health monitoring of ...  0.499939   

          u llm_green_suggested llm_confidence llm_rationale is_green_human  \
0  0.999971                                                                   
1  0.999949                                                                   
2  0.999878                                                                   

  notes  
0        
1        
2        



/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### 🤖 Part C: QLoRA Adaptation & Multi-Agent Labeling

The implementation of **QLoRA fine-tuning and the Multi-Agent System (MAS)** was conducted using the **AI Lab infrastructure**, where the training pipeline and agent orchestration are managed across several dedicated scripts and configuration files.

Because of this setup, the detailed implementation (QLoRA training, model serving, and agent coordination) is located in **separate project files** rather than directly in this notebook.

In short, this stage performs two tasks:

1. **Domain Adaptation (QLoRA)**  
   A generative LLM is fine-tuned on the `train_silver` dataset to better understand **dense patent language and Y02 climate technology classifications**.

2. **Multi-Agent Debate System (MAS)**  
   The fine-tuned model is used as the reasoning engine for a **three-agent debate system**:
   - **Advocate** argues that a claim should be classified as green  
   - **Skeptic** argues against the classification (detects greenwashing)  
   - **Judge** evaluates both arguments and outputs the final label with rationale

This system is applied to the **100 high-uncertainty claims** produced in Part B, generating structured outputs used later in the **targeted human review (HITL)** step.

### 🏁 Part D: Targeted Human Review (HITL)

Implements a simple **CLI review tool** that loads the MAS output CSV (`hitl_review.csv`) and lets a human confirm or override the **Judge** decision.

For each pending case, the script prints:
- Judge output (`is_green`, optional confidence/deadlock/Y02 hint + rationale)
- The full claim text
- Advocate and Skeptic arguments (optionally with strength scores)

The reviewer assigns a final **gold label** (`human_is_green_gold ∈ {0,1}`) and can add an optional note.  
Progress is written to `hitl_reviewed_by_me.csv` **after every decision**, so the session can be safely resumed.

The script is also robust to small column-name changes by auto-detecting the first matching column name for each required field.

In [ ]:
# --- Simple CLI tool for targeted human review (HITL) of MAS output CSV ---
# Reads MAS-produced hitl_review.csv, shows each case with arguments, and writes your decisions incrementally.

import pandas as pd
from pathlib import Path

# --- Input/Output files: IN is generated by MAS, OUT is your continuously saved review file ---
IN_CSV = "AI-lab/hitl_review.csv"          # produced by MAS
OUT_CSV = "AI-lab/hitl_reviewed_by_me.csv"  # your reviewed file (saved after each decision)

# --- Formatting: wrap long text blocks for terminal readability ---
TEXT_WRAP = 100

def wrap(s: str, width: int = 100) -> str:
    # --- Word-wrap helper: preserves paragraphs and wraps long lines at spaces ---
    s = "" if s is None else str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for para in s.split("\n"):
        if not para.strip():
            lines.append("")
            continue
        while len(para) > width:
            cut = para.rfind(" ", 0, width)
            if cut == -1:
                cut = width
            lines.append(para[:cut])
            para = para[cut:].lstrip()
        lines.append(para)
    return "\n".join(lines)

def normalize_human_label(x):
    # --- Normalize user label field into {0,1} or "" (empty = not reviewed yet) ---
    if pd.isna(x) or x == "":
        return ""
    try:
        v = int(float(x))
        if v in (0, 1):
            return v
    except Exception:
        pass
    return ""

def first_existing(df: pd.DataFrame, options):
    # --- Robustness: accept slight column name variations by returning first match ---
    for c in options:
        if c in df.columns:
            return c
    return None

def main():
    # --- Ensure MAS output exists before starting interactive review ---
    path = Path(IN_CSV)
    if not path.exists():
        raise FileNotFoundError(f"Could not find {IN_CSV}. Run MAS first to generate it.")

    # --- Load HITL cases CSV produced by MAS ---
    df = pd.read_csv(IN_CSV)

    # Try to support slight column-name variations
    # --- Column detection: makes script compatible if MAS export columns changed slightly ---
    col_claim_id = first_existing(df, ["claim_id", "doc_id", "id"])
    col_text = first_existing(df, ["text", "claim_text", "claim"])
    col_adv_arg = first_existing(df, ["advocate_argument", "adv_argument", "advocate"])
    col_ske_arg = first_existing(df, ["skeptic_argument", "ske_argument", "skeptic"])
    col_adv_strength = first_existing(df, ["adv_strength", "advocate_strength"])
    col_ske_strength = first_existing(df, ["ske_strength", "skeptic_strength"])
    col_judge_is_green = first_existing(df, ["judge_is_green", "is_green"])
    col_conf = first_existing(df, ["confidence", "judge_confidence", "conf"])
    col_deadlock = first_existing(df, ["deadlock"])
    col_y02 = first_existing(df, ["judge_y02_hint", "y02_hint"])
    col_rat = first_existing(df, ["judge_rationale", "rationale"])

    # Human columns (create if missing)
    # --- Ensure columns for your gold label + optional notes exist ---
    if "human_is_green_gold" not in df.columns:
        df["human_is_green_gold"] = ""
    if "human_note" not in df.columns:
        df["human_note"] = ""

    # --- Validate that minimum required columns exist before continuing ---
    required = [col_claim_id, col_text, col_adv_arg, col_ske_arg, col_judge_is_green]
    missing = [name for name, col in [
        ("claim_id", col_claim_id),
        ("text", col_text),
        ("advocate_argument", col_adv_arg),
        ("skeptic_argument", col_ske_arg),
        ("judge_is_green", col_judge_is_green),
    ] if col is None]

    if missing:
        raise ValueError(
            "Your HITL CSV doesn't have the expected columns.\n"
            f"Missing: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    # --- Normalize any existing labels from previous sessions (keeps progress) ---
    df["human_is_green_gold"] = df["human_is_green_gold"].apply(normalize_human_label)

    # --- Only review rows not yet labeled (empty string means pending) ---
    to_review = df[df["human_is_green_gold"].astype(str).eq("")].index.tolist()

    # --- UI instructions for the interactive labeling loop ---
    print(f"Loaded {len(df)} HITL cases from {IN_CSV}")
    print(f"Remaining to review: {len(to_review)}")
    print("Controls: [1]=green, [0]=not green, [s]=skip, [q]=quit")
    print("Tip: press ENTER = skip\n")

    # --- Iterate remaining cases and prompt you for a label ---
    for k, idx in enumerate(to_review, start=1):
        row = df.loc[idx]

        # --- Pull the fields shown to the reviewer (judge summary + claim + arguments) ---
        claim_id = row[col_claim_id]
        judge_is_green = row[col_judge_is_green]
        conf = row[col_conf] if col_conf else ""
        deadlock = row[col_deadlock] if col_deadlock else ""
        y02 = row[col_y02] if col_y02 else ""
        rat = row[col_rat] if col_rat else ""

        adv_strength = row[col_adv_strength] if col_adv_strength else ""
        ske_strength = row[col_ske_strength] if col_ske_strength else ""

        # --- Print a compact header with the model decision context ---
        print("=" * 110)
        print(f"[{k}/{len(to_review)}] claim_id={claim_id}")
        line = f"Judge: is_green={judge_is_green}"
        if conf != "":
            line += f"  conf={conf}"
        if deadlock != "":
            line += f"  deadlock={deadlock}"
        if y02 != "":
            line += f"  y02_hint={y02}"
        if adv_strength != "" or ske_strength != "":
            line += f"  (adv={adv_strength}, ske={ske_strength})"
        print(line)

        # --- Show judge rationale if available (helps faster human decision) ---
        if str(rat).strip():
            print("\nJudge rationale:")
            print(wrap(rat, TEXT_WRAP))

        # --- Show claim text and both sides' arguments for transparency ---
        print("\nCLAIM:")
        print(wrap(row[col_text], TEXT_WRAP))

        print("\nADVOCATE:")
        print(wrap(row[col_adv_arg], TEXT_WRAP))

        print("\nSKEPTIC:")
        print(wrap(row[col_ske_arg], TEXT_WRAP))
        print()

        # --- Input loop: enforce valid commands and save after each label ---
        while True:
            ans = input("Your label? [1/0/s/q] > ").strip().lower()
            if ans in ("1", "0"):
                # --- Save your gold label and optionally a note, then write file immediately ---
                df.at[idx, "human_is_green_gold"] = int(ans)
                note = input("Optional note (ENTER to skip) > ").strip()
                if note:
                    df.at[idx, "human_note"] = note
                df.to_csv(OUT_CSV, index=False)
                print(f"✅ Saved decision to {OUT_CSV}\n")
                break
            elif ans in ("s", ""):
                # --- Skip keeps the row unlabeled so it will appear next time ---
                print("⏭️  Skipped.\n")
                break
            elif ans == "q":
                # --- Quit: persist progress and exit cleanly ---
                df.to_csv(OUT_CSV, index=False)
                print(f"💾 Saved progress to {OUT_CSV}. Exiting.")
                return
            else:
                print("Please type 1, 0, s, or q.")

    # --- Final save after completing all pending cases ---
    df.to_csv(OUT_CSV, index=False)
    print(f"✅ All done. Saved {OUT_CSV}")

# --- Script entry point ---
if __name__ == "__main__":
    main()

Loaded 8 HITL cases from AI-lab/hitl_review.csv
Remaining to review: 8
Controls: [1]=green, [0]=not green, [s]=skip, [q]=quit
Tip: press ENTER = skip

[1/8] claim_id=8618928
Judge: is_green=0  conf=0.5900000000000001  deadlock=True  y02_hint=nan  (adv=7.0, ske=6.0)

Judge rationale:
The claim focuses on electronic monitoring and data transmission, which does not inherently relate
to climate mitigation/adaptation technologies.

CLAIM:
1. A method for wireless health monitoring of a locator beacon mechanically attached to a housing
of a data recorder provided to record specific performance parameters of a vehicle, comprising:
coupling a first Transponder and Sensor Module (TSM) directly to or adjacent to said locator beacon
such that at least one condition of said locator beacon or a battery of said locator beacon can be
remotely monitored with assistance of the first TSM, wherein the first TSM comprises a first
electronic circuit with a power source separate from the battery of the loca

### 🧩 Part D: Gold Dataset Assembly (MAS + Targeted HITL Merge)

Merges the full MAS outputs (`mas_labels.jsonl`) with the reviewed human decisions (`hitl_reviewed_by_me.csv`) to create the final **100-claim gold dataset**.

The script flattens the MAS JSONL into a tabular format (claim text + Judge decision + metadata such as confidence/deadlock/Y02 hint/rationale). It then applies the final labeling rule:

- **Use the human label** (`human_is_green_gold`) when it exists  
- Otherwise **fall back to the Judge label** (`judge_is_green`)

The resulting dataset adds:
- `is_green_gold` (final gold label)
- `label_source` (`human` vs `judge`) for disagreement reporting
- optional `human_note` for traceability

Exports the final file as `claims_100_gold.csv` and prints a summary of how many claims required human intervention vs automatic acceptance of the Judge decision.

In [ ]:
# --- Merge MAS outputs + targeted HITL decisions into final 100-claim gold dataset ---
# Logic: prefer human label when present; otherwise accept Judge label from MAS.

import json
import pandas as pd
from pathlib import Path

# --- Inputs: full MAS JSONL + your reviewed HITL CSV; Output: final gold CSV ---
IN_JSONL = "AI-lab/mas_labels.jsonl"
IN_HITL  = "AI-lab/hitl_reviewed_by_me.csv"   # reviewed file (or outputs/hitl_review.csv if you edited in-place)
OUT_GOLD = "AI-lab/claims_100_gold.csv"

def normalize_01(x):
    # --- Normalize labels into {0,1} or None (None = missing/invalid/unreviewed) ---
    if x is None:
        return None
    if isinstance(x, str) and x.strip() == "":
        return None
    try:
        v = int(float(x))
        return v if v in (0, 1) else None
    except Exception:
        return None

def load_jsonl(path: str):
    # --- Read JSONL file into a list of dicts (one dict per claim) ---
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def main():
    # --- Preconditions: require MAS output and reviewed HITL file to exist ---
    if not Path(IN_JSONL).exists():
        raise FileNotFoundError(f"Missing {IN_JSONL} (run MAS first).")
    if not Path(IN_HITL).exists():
        raise FileNotFoundError(
            f"Missing {IN_HITL}. Review your HITL cases first, or point IN_HITL to the correct file."
        )

    # --- Load MAS full output (structured JSON records) ---
    mas_rows = load_jsonl(IN_JSONL)

    # Flatten MAS jsonl into a simple table
    # --- Convert nested JSON structure into a flat dataframe for merging/export ---
    flat = []
    for r in mas_rows:
        claim_id = str(r.get("claim_id"))
        text = r.get("text", "")

        # --- Extract Judge fields (final decision + metadata) ---
        jud = r.get("judge", {}) or {}
        judge_is_green = normalize_01(jud.get("is_green"))
        judge_y02_hint = (jud.get("y02_hint") or "").strip()
        judge_rationale = (jud.get("rationale") or "").strip()

        flat.append({
            "claim_id": claim_id,
            "text": text,
            "judge_is_green": judge_is_green,
            "judge_y02_hint": judge_y02_hint,
            "judge_rationale": judge_rationale,
            "confidence": r.get("confidence", None),
            "deadlock": r.get("deadlock", None),
            "needs_human_review": r.get("needs_human_review", None),
        })

    df_mas = pd.DataFrame(flat)

    # Load HITL review file
    # --- Load your human-reviewed CSV and validate required columns exist ---
    df_hitl = pd.read_csv(IN_HITL)
    if "claim_id" not in df_hitl.columns:
        raise ValueError(f"{IN_HITL} must contain a 'claim_id' column.")
    if "human_is_green_gold" not in df_hitl.columns:
        raise ValueError(f"{IN_HITL} must contain 'human_is_green_gold' column.")

    # --- Normalize types and labels for reliable join/mapping ---
    df_hitl["claim_id"] = df_hitl["claim_id"].astype(str)
    df_hitl["human_is_green_gold"] = df_hitl["human_is_green_gold"].apply(normalize_01)
    if "human_note" not in df_hitl.columns:
        df_hitl["human_note"] = ""

    # Create mapping claim_id -> human label
    # --- Build fast lookup dicts for human labels and notes (avoids dataframe join complexity) ---
    human_map = dict(zip(df_hitl["claim_id"], df_hitl["human_is_green_gold"]))
    note_map  = dict(zip(df_hitl["claim_id"], df_hitl["human_note"]))

    # Merge: prefer human if present, else judge
    # --- Gold label policy: human overrides judge; otherwise accept judge output as gold ---
    gold_labels = []
    sources = []
    human_notes = []

    for _, row in df_mas.iterrows():
        cid = row["claim_id"]
        h = human_map.get(cid, None)
        h = normalize_01(h)
        if h is not None:
            gold_labels.append(h)
            sources.append("human")
            human_notes.append(note_map.get(cid, ""))
        else:
            gold_labels.append(row["judge_is_green"])
            sources.append("judge")
            human_notes.append("")

    # --- Attach merged label + provenance (useful for report: % HITL interventions) ---
    df_mas["is_green_gold"] = gold_labels
    df_mas["label_source"] = sources
    df_mas["human_note"] = human_notes

    # Sanity checks
    # --- Check for any missing gold labels (should be rare if judge fallback worked) ---
    missing_gold = df_mas["is_green_gold"].isna().sum()
    if missing_gold > 0:
        # This should not happen unless judge_is_green missing in jsonl
        print(f"WARNING: {missing_gold} rows have missing is_green_gold. Check judge outputs / jsonl parsing.")

    # Order columns nicely
    # --- Final column ordering for clean downstream training and easy inspection ---
    cols = [
        "claim_id",
        "text",
        "is_green_gold",
        "label_source",
        "judge_is_green",
        "judge_y02_hint",
        "judge_rationale",
        "human_note",
        "confidence",
        "deadlock",
        "needs_human_review",
    ]
    cols = [c for c in cols if c in df_mas.columns]
    df_out = df_mas[cols].copy()

    # --- Write gold dataset CSV (and ensure parent folder exists) ---
    Path(Path(OUT_GOLD).parent).mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUT_GOLD, index=False)
    print(f"✅ Wrote {OUT_GOLD}")

    # --- Report: how many labels came from human vs judge (key metric in assignment writeup) ---
    print(df_out["label_source"].value_counts(dropna=False).to_string())

if __name__ == "__main__":
    main()

✅ Wrote AI-lab/claims_100_gold.csv
label_source
judge    92
human     8


### ✅ Part D: Final Fine-Tuning (PatentSBERTa on Silver + Gold)

Loads the base 50k parquet and the merged **gold_100** labels (`claims_100_gold.csv`). It creates a final training label (`is_green_gold_final`) by **overriding the silver heuristic label** with the reviewed gold label whenever a claim appears in the 100-claim set.

Training data is constructed as:
- `train_silver` (70% of the 50k dataset)
- **+** the re-labeled `gold_100` subset (to force learning on the most uncertain/high-risk examples)

A **sequence classification head** is added on top of PatentSBERTa (`num_labels=2`) and the full model is fine-tuned using HuggingFace `Trainer`.

Evaluation is reported on two targets:
- `eval_silver` (proxy evaluation using silver labels, for consistency with earlier assignments)
- `gold_100` (true labels for the high-uncertainty set, to measure improvement where it matters)

Finally, the fine-tuned model + tokenizer are saved to `models_finetuned_patentsberta_green_gold100`, and the merged dataset (including `is_green_gold_final`) is exported as `patents_50k_green_with_gold_final.parquet` for reproducibility.

In [ ]:
# --- Final fine-tuning of PatentSBERTa on (train_silver + gold_100) ---
# Creates is_green_gold_final by overriding silver labels with your reviewed gold labels,
# fine-tunes a sequence classifier, evaluates on eval_silver and on gold_100, and saves artifacts.

import os
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import precision_recall_fscore_support

# -------------------- CONFIG --------------------
# --- Input data: 50k parquet + merged gold labels (100 claims) ---
PARQUET_PATH = "patents_50k_green.parquet"
GOLD100_CSV_PATH = "AI-lab/claims_100_gold.csv"

# --- Base encoder model and output paths ---
BASE_MODEL = "AI-Growth-Lab/PatentSBERTa"
OUT_DIR = "models_finetuned_patentsberta_green_gold100"
MERGED_OUT = "patents_50k_green_with_gold_final.parquet"

# Column names in parquet
# --- Column conventions from earlier scripts ---
ID_COL = "doc_id"
TEXT_COL = "text"
SPLIT_COL = "split"
SILVER_LABEL_COL = "is_green_silver"

# Gold CSV columns (from merge script)
# --- Column conventions from MAS+HITL merge export ---
GOLD_ID_COL = "claim_id"
GOLD_LABEL_COL = "is_green_gold"

# Training settings
# --- Fine-tuning hyperparams (small epochs since data is already strong/silver-heavy) ---
MAX_LEN = 256
EPOCHS = 1
LR = 2e-5
TRAIN_BS = 16
EVAL_BS = 32
SEED = 42

# -------------------- DEVICE --------------------
# --- Device selection: MPS > CUDA > CPU ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)

def clean_binary_label(x):
    # --- Parse potentially messy CSV labels into clean {0,1} or None ---
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None
    # accepts "0", "1", "0.0", "1.0", etc.
    if re.fullmatch(r"[01](\.0+)?", s):
        return int(float(s))
    return None

def compute_metrics(eval_pred):
    # --- HuggingFace Trainer metrics: binary PRF1 + accuracy from logits ---
    if hasattr(eval_pred, "predictions"):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = (preds == labels).mean()
    return {"precision": p, "recall": r, "f1": f1, "accuracy": acc}

def main():
    # --- Create output directory for model checkpoints and final export ---
    os.makedirs(OUT_DIR, exist_ok=True)

    # 1) Load base parquet (50k)
    # --- Load dataset and validate required columns exist ---
    df = pd.read_parquet(PARQUET_PATH)
    for c in [ID_COL, TEXT_COL, SPLIT_COL, SILVER_LABEL_COL]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {PARQUET_PATH}. Found: {list(df.columns)}")
    # --- Normalize id dtype so it matches gold claim_id dtype for mapping/isin ---
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

    # 2) Load gold_100 csv (from MAS + HITL merge)
    # --- Load final gold labels produced by the merge step ---
    gold = pd.read_csv(GOLD100_CSV_PATH)
    for c in [GOLD_ID_COL, GOLD_LABEL_COL]:
        if c not in gold.columns:
            raise ValueError(f"Missing column '{c}' in {GOLD100_CSV_PATH}. Found: {list(gold.columns)}")

    # --- Normalize gold IDs and labels (handles strings/floats/empties) ---
    gold[GOLD_ID_COL] = pd.to_numeric(gold[GOLD_ID_COL], errors="coerce").astype("Int64")
    gold[GOLD_LABEL_COL] = gold[GOLD_LABEL_COL].apply(clean_binary_label)

    # --- Keep only properly labeled gold rows ---
    gold_labeled = gold.dropna(subset=[GOLD_ID_COL, GOLD_LABEL_COL]).copy()
    gold_labeled[GOLD_LABEL_COL] = gold_labeled[GOLD_LABEL_COL].astype(int)

    # --- Fail fast if gold labels are missing (pipeline integrity) ---
    if len(gold_labeled) == 0:
        raise RuntimeError(
            f"No valid 0/1 labels found in {GOLD100_CSV_PATH} column '{GOLD_LABEL_COL}'."
        )

    # --- Not fatal: allow <100 if user skipped some HITL rows ---
    if len(gold_labeled) != 100:
        print(f"WARNING: expected 100 gold rows, found {len(gold_labeled)} (continuing anyway)")

    # --- Create claim_id -> gold label mapping ---
    gold_map = dict(zip(gold_labeled[GOLD_ID_COL].tolist(), gold_labeled[GOLD_LABEL_COL].tolist()))
    print("Gold labeled claim_ids:", len(gold_map))

    # 3) Create final gold label column in base dataset (silver overridden by gold)
    # --- Start with silver label, then override for the 100 claims we re-labeled ---
    df["is_green_gold_final"] = df[SILVER_LABEL_COL].astype(int)
    mask = df[ID_COL].isin(gold_map.keys())
    df.loc[mask, "is_green_gold_final"] = df.loc[mask, ID_COL].map(gold_map).astype(int)

    # 4) Build training/eval splits
    # --- Training set = original train_silver + the (re-labeled) gold_100 subset ---
    train_silver_df = df[df[SPLIT_COL] == "train_silver"].copy()
    gold_100_df = df[df[ID_COL].isin(gold_map.keys())].copy()

    train_df = pd.concat([train_silver_df, gold_100_df], ignore_index=True)
    train_df["label"] = train_df["is_green_gold_final"].astype(int)

    # --- Evaluation set: eval_silver (silver labels) + separate eval on gold_100 (true gold labels) ---
    eval_silver_df = df[df[SPLIT_COL] == "eval_silver"].copy()
    eval_silver_df["label"] = eval_silver_df[SILVER_LABEL_COL].astype(int)

    gold_eval_df = gold_100_df.copy()
    gold_eval_df["label"] = gold_eval_df["is_green_gold_final"].astype(int)

    # --- Logging split sizes for the report/debugging ---
    print("\nSizes:")
    print("train_silver:", len(train_silver_df))
    print("gold_100_final:", len(gold_100_df))
    print("train_final:", len(train_df))
    print("eval_silver:", len(eval_silver_df))

    # 5) Convert to HF Datasets
    # --- Convert pandas -> HF Dataset for Trainer API ---
    train_ds = Dataset.from_pandas(train_df[[TEXT_COL, "label"]], preserve_index=False)
    eval_ds = Dataset.from_pandas(eval_silver_df[[TEXT_COL, "label"]], preserve_index=False)
    gold_ds = Dataset.from_pandas(gold_eval_df[[TEXT_COL, "label"]], preserve_index=False)

    # 6) Tokenize
    # --- Tokenize texts (same truncation length used in baseline) ---
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    def tok(batch):
        return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

    train_ds = train_ds.map(tok, batched=True, remove_columns=[TEXT_COL])
    eval_ds = eval_ds.map(tok, batched=True, remove_columns=[TEXT_COL])
    gold_ds = gold_ds.map(tok, batched=True, remove_columns=[TEXT_COL])

    # --- Dynamic padding per batch (more efficient than fixed padding) ---
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 7) Model
    # --- Add classification head on top of PatentSBERTa (2 labels: non-green/green) ---
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2).to(DEVICE)

    # 8) Training args (Solution 1: no evaluation_strategy for older transformers)
    # --- Trainer settings: minimal logging/saving; fp16 disabled for MPS stability ---
    args = TrainingArguments(
        output_dir=OUT_DIR,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        seed=SEED,
        report_to="none",
        fp16=False,
        logging_steps=20,
        save_steps=500,
        save_total_limit=2,
    )

    # --- Trainer wires together model, data, collator, and metrics ---
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,  # used only if you call trainer.evaluate(eval_ds)
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    # 9) Train
    # --- Fine-tune full encoder + classification head on train_final ---
    trainer.train()

    # 10) Evaluate
    # --- Evaluate on eval_silver (silver proxy test) and separately on gold_100 (true labels for your 100 risky claims) ---
    print("\n=== Final Model: eval_silver (silver labels) ===")
    m_eval = trainer.evaluate(eval_dataset=eval_ds)
    print(m_eval)

    print("\n=== Final Model: gold_100 (Gold labels) ===")
    m_gold = trainer.evaluate(eval_dataset=gold_ds)
    print(m_gold)

    # 11) Save model/tokenizer
    # --- Persist fine-tuned model + tokenizer for later inference/reporting ---
    trainer.save_model(OUT_DIR)
    tokenizer.save_pretrained(OUT_DIR)

    # 12) Save merged parquet
    # --- Save dataset with the final label column for reproducibility/inspection ---
    df.to_parquet(MERGED_OUT, index=False)
    print(f"\nSaved merged dataset: {MERGED_OUT}")
    print(f"Saved fine-tuned model: {OUT_DIR}")

if __name__ == "__main__":
    main()

Using device: mps
Gold labeled claim_ids: 100

Sizes:
train_silver: 35000
gold_100_final: 100
train_final: 35100
eval_silver: 5000


Map:   0%|          | 0/35100 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

{'eval_loss': 0.41487276554107666, 'eval_precision': 0.8235294117647058, 'eval_recall': 0.7784, 'eval_f1': 0.8003290150113099, 'eval_accuracy': 0.8058, 'eval_runtime': 185.9854, 'eval_samples_per_second': 26.884, 'eval_steps_per_second': 0.844, 'epoch': 1.0}

=== Final Model: gold_100 (Gold labels) ===


/opt/miniconda3/envs/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.8000638484954834, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_accuracy': 0.55, 'eval_runtime': 3.7896, 'eval_samples_per_second': 26.388, 'eval_steps_per_second': 1.056, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved merged dataset: patents_50k_green_with_gold_final.parquet
Saved fine-tuned model: models_finetuned_patentsberta_green_gold100


### 📊 Part E: Standalone Evaluation + Prediction Export

Loads the base dataset (`patents_50k_green.parquet`) and recreates the same **gold override mapping** used during training by loading `claims_100_gold.csv` and rebuilding `is_green_gold_final` (silver label overridden for the 100 reviewed claim IDs).

It then loads the **final fine-tuned PatentSBERTa sequence classifier** from `models_finetuned_patentsberta_green_gold100` and runs batched inference to compute `p_green = P(green=1)` using softmax over logits.

Main evaluation is performed on:
- `eval_silver` (using `is_green_silver` as proxy ground-truth)

Reported metrics include accuracy, precision, recall, F1, confusion matrix, and a full classification report.

For analysis and reporting, the script exports per-row outputs for `eval_silver` to:
`final_model_eval_silver_predictions.csv`  
including `doc_id`, text, true label, `p_green`, and predicted label (`p_green >= 0.5`).

As an additional sanity check, it can also evaluate performance on the **gold_100** subset using `is_green_gold_final`.

In [ ]:
# --- Standalone evaluation script for the final fine-tuned PatentSBERTa classifier ---
# Evaluates on eval_silver (silver labels) and optionally on gold_100 (gold_final labels),
# and exports per-row predictions/probabilities to CSV.

import re
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report
)

# ---------------- Config ----------------
# --- Inputs: base parquet + gold CSV to recreate gold_final mapping; model dir is fine-tuned output ---
PARQUET_PATH = "patents_50k_green.parquet"
GOLD100_CSV_PATH = "AI-lab/claims_100_gold.csv"

MODEL_DIR = "models_finetuned_patentsberta_green_gold100"

# --- Column conventions used across pipeline ---
ID_COL = "doc_id"
TEXT_COL = "text"
SPLIT_COL = "split"
SILVER_LABEL_COL = "is_green_silver"

# --- Gold CSV column names (from merge step) ---
GOLD_ID_COL = "claim_id"
GOLD_LABEL_COL = "is_green_gold"

# --- Tokenization/inference params ---
MAX_LEN = 256
BATCH_SIZE = 64

# --- Output: save eval_silver predictions for analysis/reporting ---
OUT_EVAL_CSV = "final_model_eval_silver_predictions.csv"

# ---------------- Device ----------------
# --- Device selection: MPS > CUDA > CPU ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


def clean_binary_label(x):
    # --- Parse possibly messy CSV labels into clean {0,1} or None ---
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None
    if re.fullmatch(r"[01](\.0+)?", s):
        return int(float(s))
    return None


def build_gold_map():
    # --- Load gold CSV and build dict: claim_id -> gold label ---
    gold = pd.read_csv(GOLD100_CSV_PATH)
    for c in [GOLD_ID_COL, GOLD_LABEL_COL]:
        if c not in gold.columns:
            raise ValueError(f"Missing column '{c}' in {GOLD100_CSV_PATH}. Found: {list(gold.columns)}")

    # --- Normalize types and labels for stable mapping ---
    gold[GOLD_ID_COL] = pd.to_numeric(gold[GOLD_ID_COL], errors="coerce").astype("Int64")
    gold[GOLD_LABEL_COL] = gold[GOLD_LABEL_COL].apply(clean_binary_label)

    gold_labeled = gold.dropna(subset=[GOLD_ID_COL, GOLD_LABEL_COL]).copy()
    gold_labeled[GOLD_LABEL_COL] = gold_labeled[GOLD_LABEL_COL].astype(int)

    if len(gold_labeled) == 0:
        raise RuntimeError(f"No valid 0/1 labels found in {GOLD100_CSV_PATH} column '{GOLD_LABEL_COL}'.")

    gold_map = dict(zip(gold_labeled[GOLD_ID_COL].tolist(), gold_labeled[GOLD_LABEL_COL].tolist()))
    print("Gold labeled claim_ids:", len(gold_map))
    return gold_map


def attach_is_green_gold_final(df: pd.DataFrame, gold_map: dict) -> pd.DataFrame:
    # --- Recreate training-time gold_final label: start from silver, override selected claim_ids with gold ---
    df = df.copy()
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")
    df["is_green_gold_final"] = df[SILVER_LABEL_COL].astype(int)

    mask = df[ID_COL].isin(gold_map.keys())
    df.loc[mask, "is_green_gold_final"] = df.loc[mask, ID_COL].map(gold_map).astype(int)
    return df


@torch.no_grad()
def predict_p_green(texts, tokenizer, model, batch_size=BATCH_SIZE, max_len=MAX_LEN, device=DEVICE):
    # --- Batched inference: return probability of class=1 (green) for each text ---
    model.eval()
    probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)

        # --- Convert logits -> p(class=1) using softmax ---
        logits = out.logits
        p1 = torch.softmax(logits, dim=-1)[:, 1]
        probs.append(p1.detach().cpu().numpy())

    return np.concatenate(probs, axis=0)


def print_metrics(y_true, y_pred, name):
    # --- Print standard classification metrics + confusion matrix and full report ---
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n=== {name} ===")
    print(f"n        : {len(y_true)}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print("Confusion matrix [[TN, FP],[FN, TP]]:")
    print(cm)
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm}


def main():
    # 1) Load parquet
    # --- Load dataset and validate schema ---
    df = pd.read_parquet(PARQUET_PATH)
    for c in [ID_COL, TEXT_COL, SPLIT_COL, SILVER_LABEL_COL]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {PARQUET_PATH}. Found: {list(df.columns)}")

    # 2) Recreate gold_final like in training
    # --- Ensure evaluation uses identical gold override logic as training ---
    gold_map = build_gold_map()
    df = attach_is_green_gold_final(df, gold_map)

    # 3) Load fine-tuned model
    # --- Load tokenizer + sequence classifier from the fine-tuned model directory ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)

    # 4) Evaluate on eval_silver (this is your "test" split)
    # --- Main reported metric: eval_silver uses silver labels as proxy ground-truth ---
    eval_df = df[df[SPLIT_COL] == "eval_silver"].dropna(subset=[TEXT_COL]).copy()
    if len(eval_df) == 0:
        raise ValueError("Fandt ingen rækker med split=='eval_silver'.")

    y_true = eval_df[SILVER_LABEL_COL].astype(int).values  # "ground truth" for eval_silver
    texts = eval_df[TEXT_COL].astype(str).tolist()

    # --- Predict probabilities and threshold at 0.5 for hard labels ---
    p_green = predict_p_green(texts, tokenizer, model)
    y_pred = (p_green >= 0.5).astype(int)

    metrics = print_metrics(y_true, y_pred, "FINAL MODEL on eval_silver (silver labels)")

    # 5) Save CSV
    # --- Export per-row predictions for debugging/error analysis and report figures ---
    out = pd.DataFrame({
        ID_COL: eval_df[ID_COL].astype("Int64").values,
        TEXT_COL: eval_df[TEXT_COL].values,
        "y_true": y_true,
        "p_green": p_green,
        "y_pred": y_pred,
    })
    out.to_csv(OUT_EVAL_CSV, index=False)
    print(f"\nSaved: {OUT_EVAL_CSV}")

    # 6) Optional: evaluate on gold_100 only (for sanity; NOT the official score)
    # --- Secondary check: how the final classifier aligns with your gold relabeling subset ---
    gold_df = df[df[ID_COL].isin(gold_map.keys())].dropna(subset=[TEXT_COL]).copy()
    if len(gold_df) > 0:
        y_true_g = gold_df["is_green_gold_final"].astype(int).values
        texts_g = gold_df[TEXT_COL].astype(str).tolist()
        p_g = predict_p_green(texts_g, tokenizer, model)
        y_pred_g = (p_g >= 0.5).astype(int)
        _ = print_metrics(y_true_g, y_pred_g, "FINAL MODEL on gold_100 (gold_final labels)")

if __name__ == "__main__":
    main()

Using device: mps
Gold labeled claim_ids: 100


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting: 100%|██████████| 79/79 [02:26<00:00,  1.86s/it]



=== FINAL MODEL on eval_silver (silver labels) ===
n        : 5000
Accuracy : 0.8058
Precision: 0.8235
Recall   : 0.7784
F1       : 0.8003
Confusion matrix [[TN, FP],[FN, TP]]:
[[2083  417]
 [ 554 1946]]

Classification report:
              precision    recall  f1-score   support

           0     0.7899    0.8332    0.8110      2500
           1     0.8235    0.7784    0.8003      2500

    accuracy                         0.8058      5000
   macro avg     0.8067    0.8058    0.8057      5000
weighted avg     0.8067    0.8058    0.8057      5000


Saved: final_model_eval_silver_predictions.csv


Predicting: 100%|██████████| 2/2 [00:03<00:00,  1.94s/it]


=== FINAL MODEL on gold_100 (gold_final labels) ===
n        : 100
Accuracy : 0.5500
Precision: 0.0000
Recall   : 0.0000
F1       : 0.0000
Confusion matrix [[TN, FP],[FN, TP]]:
[[55 45]
 [ 0  0]]

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.5500    0.7097       100
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.5500       100
   macro avg     0.5000    0.2750    0.3548       100
weighted avg     1.0000    0.5500    0.7097       100

